In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
import timeit

In [8]:
import pandas as pd

file_path = "data/household_power_consumption.txt"

#зчитування усіх колонок як строки спочатку
df = pd.read_csv(
    file_path,
    sep=';',
    na_values='?',
    low_memory=False
)

#об'єдную Date та Time в одну колонку DateTime
df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')

#видаляю старі колонки Date та Time
df = df.drop(columns=['Date','Time'])

df.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime
0,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


In [10]:
#перевірка пропусків
print(df.isna().sum())

#перетворення колонки у float
num_cols = ['Global_active_power','Global_reactive_power','Voltage',
            'Global_intensity','Sub_metering_1','Sub_metering_2','Sub_metering_3']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

#заповнення пропусків 
df.fillna(df.mean(), inplace=True)

df.dtypes

Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
DateTime                 0
dtype: int64


Global_active_power             float64
Global_reactive_power           float64
Voltage                         float64
Global_intensity                float64
Sub_metering_1                  float64
Sub_metering_2                  float64
Sub_metering_3                  float64
DateTime                 datetime64[us]
dtype: object

In [11]:
def high_power_usage(df, threshold=5):
    result = df[df['Global_active_power'] > threshold]
    print(f"Записів з активною потужністю > {threshold} кВт: {len(result)}")
    return result

#приклад виклику
high_power_df = high_power_usage(df)
high_power_df.head()

Записів з активною потужністю > 5 кВт: 17547


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime
1,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
11,5.412,0.470,232.78,23.2,0.0,1.0,17.0,2006-12-16 17:35:00
12,5.224,0.478,232.99,22.4,0.0,1.0,16.0,2006-12-16 17:36:00


In [12]:
def current_range_devices(df, low=19, high=20):
    #фільтр по Global_intensity
    filtered = df[(df['Global_intensity'] >= low) & (df['Global_intensity'] <= high)]
    
    #фільтр по приладах: пральна + холодильник > бойлер + кондиціонер
    filtered = filtered[
        (filtered['Sub_metering_1'] + filtered['Sub_metering_2'] >
         filtered['Sub_metering_3'] + filtered['Global_active_power'] - 
         (filtered['Sub_metering_1']+filtered['Sub_metering_2']+filtered['Sub_metering_3']))
    ]
    
    print(f"Записів по струму {low}-{high} А з умовою приладів: {len(filtered)}")
    return filtered

#приклад виклику
current_df = current_range_devices(df)
current_df.head()

Записів по струму 19-20 А з умовою приладів: 5729


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime
45,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00
52,4.524,0.076,234.20,19.6,0.0,9.0,17.0,2006-12-16 18:16:00
460,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00
464,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00
475,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00


In [13]:
def random_sample_avg(df, n=500000):
    sample = df.sample(n=min(n, len(df)), random_state=42)
    #групи споживання: Sub_metering_1,2,3
    means = sample[['Sub_metering_1','Sub_metering_2','Sub_metering_3']].mean()
    print("Середні величини для трьох груп споживання:")
    display(means)
    return sample

#виклик
sample_df = random_sample_avg(df)

Середні величини для трьох груп споживання:


Sub_metering_1    1.109002
Sub_metering_2    1.283718
Sub_metering_3    6.384638
dtype: float64

In [14]:
def evening_usage(df):
    df['hour'] = df['DateTime'].dt.hour
    filtered = df[(df['hour'] >= 18) & (df['Global_active_power'] > 6)]
    
    #вибираю ті, де група 2 найбільша 
    filtered = filtered[filtered['Sub_metering_2'] == filtered[['Sub_metering_1','Sub_metering_2','Sub_metering_3']].max(axis=1)]
    
    #ділю на дві половини
    mid = len(filtered)//2
    first_half = filtered.iloc[:mid:3]   # кожен 3-й
    second_half = filtered.iloc[mid::4]  # кожен 4-й
    final = pd.concat([first_half, second_half])
    print(f"Відфільтрованих записів після 18:00 з умовою групи 2: {len(final)}")
    return final

#виклик
evening_df = evening_usage(df)
evening_df.head()

Відфільтрованих записів після 18:00 з умовою групи 2: 350


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime,hour
41,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00,18
44,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00,18
17494,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00,20
17498,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00,21
17501,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00,21


In [15]:
def normalize_standardize(df, cols=['Global_active_power','Voltage','Global_intensity']):
    scaler_std = StandardScaler()
    scaler_min = MinMaxScaler()
    
    df_std = scaler_std.fit_transform(df[cols])
    df_min = scaler_min.fit_transform(df[cols])
    
    print("Стандартизовані дані:")
    display(pd.DataFrame(df_std, columns=[f"{c}_std" for c in cols]).head())
    
    print("Нормалізовані дані:")
    display(pd.DataFrame(df_min, columns=[f"{c}_min" for c in cols]).head())
    
    return df_std, df_min

#виклик
std_df, min_df = normalize_standardize(df)

Стандартизовані дані (перші 5 рядків):


,Global_active_power_std,Voltage_std,Global_intensity_std
0,2.967025,-0.110675,3.110448
1,4.048676,-0.155539,4.145002
2,4.061913,-0.168145,4.145002
3,4.075150,-0.151460,4.145002
4,2.447000,-0.079529,2.525700


Нормалізовані дані (перші 5 рядків):


,Global_active_power_min,Voltage_min,Global_intensity_min
0,0.379069,0.924021,0.380165
1,0.481928,0.919260,0.475207
2,0.483186,0.917922,0.475207
3,0.484445,0.919693,0.475207
4,0.329617,0.927326,0.326446


In [16]:
def correlations(df, col1, col2):
    pearson = df[col1].corr(df[col2], method='pearson')
    spearman = df[col1].corr(df[col2], method='spearman')
    print(f"Коефіцієнт Пірсона між {col1} та {col2}: {pearson}")
    print(f"Коефіцієнт Спірмена між {col1} та {col2}: {spearman}")

#виклик
correlations(df, 'Global_active_power','Global_intensity')

Коефіцієнт Пірсона між Global_active_power та Global_intensity: 0.9989028878670703
Коефіцієнт Спірмена між Global_active_power та Global_intensity: 0.9955437183011423


In [20]:
from sklearn.preprocessing import OneHotEncoder

def one_hot_encode(df, col='hour'):
    encoder = OneHotEncoder(sparse_output=False)  
    encoded = encoder.fit_transform(df[[col]])
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out([col]))
    print(f"One Hot Encoding для {col} (перші 5 рядків):")
    display(encoded_df.head())
    return encoded_df

#виклик
encoded_hour = one_hot_encode(df)

One Hot Encoding для hour (перші 5 рядків):


,hour_0,hour_1,hour_2,hour_3,hour_4,hour_5,hour_6,hour_7,hour_8,hour_9,...,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
